# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Đã chạy trên Colab G4 (RTX PRO 6000 Blackwell 96 GB) qua colab CLI; T4 cũng chạy được với cùng mã.
!nvidia-smi -L
!pip -q install -r requirements-colab.txt kagglehub
!pip uninstall -y -q torchao   # torchao 0.10 có sẵn trên Colab làm peft lỗi khi gắn LoRA vào Linear fp16 của ViT

In [ ]:
# Dataset công khai: kagglehub tải được không cần khoá. Nếu cần khoá, dùng Colab Secrets, KHÔNG viết khoá vào notebook.
import kagglehub
SRC = kagglehub.dataset_download("lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes")
print(SRC)

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --source {SRC}
!python -m pytest -q

In [ ]:
import json, sys
from collections import Counter
sys.path.insert(0, ".")
from scripts.labcommon import read_jsonl
from lab10.viz import draw
from scripts.score import _gt_boxes

splits = {s: read_jsonl(f"data/splits/{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    print(s, len(rows), "ảnh", Counter(o["label"] for r in rows for o in r["objects"]))
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sess["train"] & sess["val"] or sess["train"] & sess["test"] or sess["val"] & sess["test"])
print("không session nào nằm ở hai split")
for i, r in enumerate(splits["train"][:6]):
    draw(r, [], "data", f"work/sample_{i}.jpg")   # box GT màu xanh

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab10.data import yolo_to_box, objects_to_answer, prompt_for
r = splits["train"][0]
ans = objects_to_answer(r["objects"]); print(ans)
print(prompt_for("norm1000"))
# kiểm tra đi-về: JSON 0–1000 → pixel phải trùng box YOLO gốc (sai số làm tròn ≤ 0,4 px)
for b in json.loads(ans):
    print([round(v / 1000 * (640 if i % 2 == 0 else 360), 1) for i, v in enumerate(b["bbox_2d"])])

## 3. E0: zero-shot

In [ ]:
# lab10/vlm.py: load_model() (NF4, skip "model.visual" + "lm_head") và predict() (batch 16, greedy, 256 token)
!python run_exp.py e0

In [ ]:
# run_exp.py e0 đã chấm; chạy lại khi cần:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from lab10 import vlm
model, proc = vlm.load_model()
item = vlm.encode_example(proc, vlm.open_image(r), prompt_for(), ans)
img_tok = proc.tokenizer.convert_tokens_to_ids("<|image_pad|>")
print("image tokens:", int((item["input_ids"] == img_tok).sum()))          # 220
lab = item["labels"][0]
print(proc.tokenizer.decode(lab[lab != -100]))                              # JSON + <|im_end|>

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
for targets, proj in (("llm", False), ("llm", True), ("llm+vit", True)):
    m, matched, extra = vlm.apply_setup(model, targets=targets, train_projector=proj)
    print(targets, proj, len(matched), vlm.count_params(m)[0])     # 252/33.0M, 252/142.1M, 348/148.4M
    model = m.unload()
    for n, p in model.named_parameters():
        if any(k in n for k in vlm.PROJECTOR_KEYS):
            p.data = p.data.half()

## 6. Vòng huấn luyện

In [ ]:
# lab10/vlm.py: train() — AdamW lr 2e-4, warmup 5% + cosine, GradScaler fp16, grad ckpt,
# batch 2 × accum 4, 100 bước; lưu adapter + projector.pt (E2/E3) mỗi 50 bước; ghi run.json
# run_exp.py: train → suy luận test → scripts/score.py --meta ...
!python run_exp.py e1
!python run_exp.py e2
!python run_exp.py e3

## 7. Suy luận và chấm E1–E3

In [ ]:
# Kiểm tra đường nạp lại: model gốc 4-bit → adapter → projector.pt (lab10/vlm.py: load_trained)
m2, p2 = vlm.load_trained("work/adapters/e3")
vlm.predict(m2, p2, vlm.load_split("val", limit=40), "work/e3_reload_val40.jsonl")
!python scripts/score.py --pred work/e3_reload_val40.jsonl --name E3-reload --split val --out work/debug/e3_reload.json
!python lab10/viz.py --pred work/e3.jsonl --prefix e3
!python lab10/viz.py --pred work/e0.jsonl --prefix e0 --n 2

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: ablation data_size trên setup tốt nhất (E3): 10% ảnh train và 100% (= E3)
!python run_exp.py e4 --base e3 --data-size 0.1 --tag data_size_0.1
!python scripts/score.py --pred work/e3.jsonl --name "E4 data_size_1.0" --out report/evidence/e4_data_size_1.0.json     --meta trainable_params=148426752 --meta peak_vram_gb=8.31 --meta train_minutes=2.29   # từ work/adapters/e3/run.json